In [6]:
!pip install streamlit
import streamlit as st
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ------------------ PAGE CONFIG ------------------
st.set_page_config(page_title="AnimeFlix 🎌", layout="wide")

# ------------------ CUSTOM CSS (ANIMATION + UI) ------------------
st.markdown("""
<style>
body {
    background-color: #0e1117;
}

.title {
    font-size: 50px;
    font-weight: bold;
    color: #ff4b4b;
    text-align: center;
    animation: fadeIn 2s;
}

.card {
    background-color: #1c1f26;
    padding: 15px;
    border-radius: 15px;
    transition: transform 0.3s;
}

.card:hover {
    transform: scale(1.05);
}

@keyframes fadeIn {
    from {opacity: 0;}
    to {opacity: 1;}
}
</style>
""", unsafe_allow_html=True)

st.markdown('<div class="title">🎌 AnimeFlix Recommendation System</div>', unsafe_allow_html=True)

# ------------------ LOAD DATA ------------------
@st.cache_data
def load_data():
    df = pd.read_csv("/content/anime (1).csv")
    df = df[df['type'] == 'Movie']
    df['genre'] = df['genre'].fillna('')
    df['rating'] = df['rating'].fillna(df['rating'].mean())
    return df

df = load_data()

# ------------------ ML MODEL ------------------
@st.cache_resource
def compute_similarity(data):
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(data['genre'])
    return cosine_similarity(tfidf_matrix, tfidf_matrix)

cosine_sim = compute_similarity(df)

indices = pd.Series(df.index, index=df['name']).drop_duplicates()

def recommend(anime_name, top_n=6):
    if anime_name not in indices:
        return pd.DataFrame()
    idx = indices[anime_name]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    anime_indices = [i[0] for i in sim_scores]
    return df.iloc[anime_indices]

# ------------------ API FUNCTION ------------------
def fetch_details(name):
    url = f"https://api.jikan.moe/v4/anime?q={name}&limit=1"
    try:
        res = requests.get(url).json()
        if res['data']:
            anime = res['data'][0]
            image = anime['images']['jpg']['image_url']
            trailer = anime['trailer']['url']
            return image, trailer
    except:
        pass
    return None, None

# ------------------ SEARCH ------------------
anime_list = df['name'].sort_values().unique()
selected = st.selectbox("🔍 Search Anime", anime_list)

top_n = st.slider("Number of Recommendations", 3, 12, 6)

# ------------------ BUTTON ------------------
if st.button("🚀 Show Recommendations"):

    results = recommend(selected, top_n)

    st.subheader("🎯 Recommended For You")

    cols = st.columns(3)

    for i, (_, row) in enumerate(results.iterrows()):
        image, trailer = fetch_details(row['name'])

        with cols[i % 3]:
            st.markdown('<div class="card">', unsafe_allow_html=True)

            if image:
                st.image(image, use_column_width=True)

            st.markdown(f"### 🎬 {row['name']}")
            st.write(f"⭐ {row['rating']}")
            st.write(f"🎭 {row['genre']}")

            if trailer:
                st.video(trailer)
            else:
                yt = f"https://www.youtube.com/results?search_query={row['name']}+trailer"
                st.markdown(f"[▶️ Watch Trailer]({yt})")

            st.markdown('</div>', unsafe_allow_html=True)

# ------------------ TRENDING SECTION ------------------
st.subheader("🔥 Trending Anime")

top = df.sort_values(by='rating', ascending=False).head(6)

cols = st.columns(3)

for i, (_, row) in enumerate(top.iterrows()):
    image, _ = fetch_details(row['name'])

    with cols[i % 3]:
        st.markdown('<div class="card">', unsafe_allow_html=True)

        if image:
            st.image(image)

        st.write(f"**{row['name']}**")
        st.write(f"⭐ {row['rating']}")

        st.markdown('</div>', unsafe_allow_html=True)

2026-05-15 15:09:56.580 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 15:09:56.581 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 15:09:56.583 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 15:09:56.584 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 15:09:56.587 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 15:09:56.588 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 15:09:56.589 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 15:09:56.590 No runtime found, using MemoryCacheStorageManager
2026-05-15 15:09:56.607 Thread 'MainThread':